In [1]:
%pip install nltk spacy corenlp

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Text Representation

In [2]:
import pandas as pd 
import nltk
import spacy
import corenlp

In [3]:
import spacy
from spacy.cli import download as spacy_download

model_name = 'en_core_web_md'
try:
    nlp = spacy.load(model_name)
except OSError:
    print(f"{model_name} not found. Downloading...")
    spacy_download(model_name)
    nlp = spacy.load(model_name)

In [4]:
metadata_df = pd.read_parquet('meta_video_games.parquet') # very large file

In [5]:
# load train and test data csv files
train_df = pd.read_csv('csv_files/train_video_games.csv')
test_df = pd.read_csv('csv_files/test_video_games.csv')
# compare the number of common items in train and test data
common_items = set(train_df['item_id']).intersection(set(test_df['item_id']))
print(f'Number of common items in train and test data: {len(common_items)}')

Number of common items in train and test data: 927


I only want the items relevant to our train and test set. Thus I am picking the itersection of items in train and test. 

In [6]:
# only extract the metadata for the common items in train and test data
common_metadata_df = metadata_df[metadata_df['item_id'].isin(common_items)]
common_metadata_df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 927 entries, 864 to 136488
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   item_id         927 non-null    object 
 1   title           927 non-null    object 
 2   description     927 non-null    object 
 3   features        927 non-null    object 
 4   categories      927 non-null    object 
 5   details         927 non-null    object 
 6   rating_number   927 non-null    int64  
 7   average_rating  927 non-null    float64
dtypes: float64(1), int64(1), object(6)
memory usage: 65.2+ KB


In [7]:
# select the column description from the common metadata dataframe
descriptions = common_metadata_df['description'].tolist()
element = descriptions[0] # its a np.ndarray, we need to convert it to string
type(descriptions[0])

numpy.ndarray

Apply appropriate preprocessing to clean up the data, for example: tokenization,
transformation to lowercase, stopword removal, 6 or stemming. Motivate
your preprocessing choices and report the vocabulary size before and after
preprocessing. 

In [8]:
import nltk

for resource in ("punkt_tab", "punkt"):
    nltk.download(resource, quiet=True)

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\armin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
# using the libraries pre process the descriptions
# lower case the descriptions
import string
# can I manipulate in place because downstream I need the asscociated item_id and description, so I will create a new list of clean descriptions
descriptions = common_metadata_df['description'].astype(str).tolist()
# motivate my pre processing steps by understanding the size, and properties of the descriptions
print(f'Number of descriptions: {len(descriptions)}')
print(f'Average length of descriptions: {round(sum(len(desc) for desc in descriptions) / len(descriptions),3)}')
print(f'Maximum length of descriptions: {max(len(desc) for desc in descriptions)}')
print(f'Minimum length of descriptions: {min(len(desc) for desc in descriptions)}')

print(descriptions[0][:300])
# check vocab size 
vocab = set()
for desc in descriptions:
    vocab.update(desc.split())
print(f'Vocabulary size: {len(vocab)}')


Number of descriptions: 927
Average length of descriptions: 2485.95
Maximum length of descriptions: 20881
Minimum length of descriptions: 2
['Amazon.com' 'Long recognized as role-playing games par excellence, the'
 'Final Fantasy'
 'series gets a technological makeover in this installment (and series debut on the PlayStation). Shedding the two-dimensional graphics and limited sound capabilities of its predecessors,'
 'Final Fantasy VII'
Vocabulary size: 40011


18,618 words seems managable 

In [10]:
clean_descriptions = [desc.lower() for desc in descriptions]
# remove punctuation from the descriptions
clean_descriptions = [desc.translate(str.maketrans('', '', string.punctuation)) for desc in clean_descriptions]
# tokenize the descriptions using nltk
tokenized_descriptions = [nltk.word_tokenize(desc) for desc in clean_descriptions]

tokenized_descriptions[0][:300]

['amazoncom',
 'long',
 'recognized',
 'as',
 'roleplaying',
 'games',
 'par',
 'excellence',
 'the',
 'final',
 'fantasy',
 'series',
 'gets',
 'a',
 'technological',
 'makeover',
 'in',
 'this',
 'installment',
 'and',
 'series',
 'debut',
 'on',
 'the',
 'playstation',
 'shedding',
 'the',
 'twodimensional',
 'graphics',
 'and',
 'limited',
 'sound',
 'capabilities',
 'of',
 'its',
 'predecessors',
 'final',
 'fantasy',
 'vii',
 'features',
 'lush',
 '3d',
 'graphics',
 'beautifully',
 'animated',
 'movie',
 'sequences',
 'and',
 'soundtrackquality',
 'music',
 'coupled',
 'with',
 'the',
 'games',
 'intricate',
 'storyline',
 'endearing',
 'characters',
 'and',
 'immense',
 'yet',
 'highly',
 'imaginative',
 'world',
 'these',
 'new',
 'advancements',
 'make',
 'for',
 'a',
 'quite',
 'an',
 'engrossing',
 'experience',
 'the',
 'story',
 'of',
 'final',
 'fantasy',
 'vii',
 'centers',
 'around',
 'a',
 'solider',
 'named',
 'cloud',
 'strife',
 'who',
 'joins',
 'forces',
 'with',

In [11]:
# remove stop words from the descriptions
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
processed_descriptions = []
for desc in tokenized_descriptions:
    processed_descriptions.append([word for word in desc if word not in stop_words])
print(processed_descriptions[0][:300])
# calculate the new vocabulary size after removing stop words
vocab = set()
for desc in processed_descriptions:
    for word in desc:
        vocab.add(word)
print(f'Vocabulary size after removing stop words: {len(vocab)}')

['amazoncom', 'long', 'recognized', 'roleplaying', 'games', 'par', 'excellence', 'final', 'fantasy', 'series', 'gets', 'technological', 'makeover', 'installment', 'series', 'debut', 'playstation', 'shedding', 'twodimensional', 'graphics', 'limited', 'sound', 'capabilities', 'predecessors', 'final', 'fantasy', 'vii', 'features', 'lush', '3d', 'graphics', 'beautifully', 'animated', 'movie', 'sequences', 'soundtrackquality', 'music', 'coupled', 'games', 'intricate', 'storyline', 'endearing', 'characters', 'immense', 'yet', 'highly', 'imaginative', 'world', 'new', 'advancements', 'make', 'quite', 'engrossing', 'experience', 'story', 'final', 'fantasy', 'vii', 'centers', 'around', 'solider', 'named', 'cloud', 'strife', 'joins', 'forces', 'avalanche', 'group', 'resistance', 'fighters', 'take', 'evil', 'megacorporation', 'known', 'shinra', 'fate', 'world', 'hangs', 'balance', 'course', 'truly', 'epic', 'scope', 'fourdisc', 'game', 'requires', 'considerable', 'amount', 'time', 'completethis', 

In [12]:
# merge back with coresponding item_id and description
processed_metadata_df = common_metadata_df.copy()
processed_metadata_df['processed_description'] = [' '.join(desc) for desc in processed_descriptions]
processed_metadata_df.head(3)

,item_id,title,description,features,categories,details,rating_number,average_rating,processed_description
864,B00000JRSB,Final Fantasy VII - PlayStation,"[Amazon.com, Long recognized as role-playing g...","[1 Player, RPG, 3 Disc Set, Excellent graphics...","[Video Games, Legacy Systems, PlayStation Syst...","{'': None, 'AC Adapter Current': None, 'Access...",1631,4.7,amazoncom long recognized roleplaying games pa...
1312,B00001X50M,Metal Gear Solid,"[Product description, You are Snake, a governm...","[Lightly Armed and facing an army of foes, Sna...","[Video Games, Legacy Systems, PlayStation Syst...","{'': None, 'AC Adapter Current': None, 'Access...",638,4.6,product description snake government agent mis...
1354,B00001XDUB,Silent Hill,"[Product Description, In, Silent Hill, , you a...",[],"[Video Games, Legacy Systems, PlayStation Syst...","{'': None, 'AC Adapter Current': None, 'Access...",399,4.4,product description silent hill assume role wi...


### TF-IDF implementation needs work

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Use the cleaned text column
# use title and processed description for the corpus, we will concatenate them together
processed_metadata_df['combined_text'] = processed_metadata_df['title'].fillna('') + ' ' + processed_metadata_df['processed_description'].fillna('')
corpus = processed_metadata_df["combined_text"].fillna("")

# Build TF-IDF representation of the description corpus 
# it will create a sparse matrix where rows correspond to items and columns correspond to terms,
# with TF-IDF values as entries 
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

# Keep item_id aligned with each TF-IDF row
item_ids = processed_metadata_df["item_id"].tolist()
feature_names = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape)  # (n_items, n_terms)
print("Number of non-zero entries:", tfidf_matrix.nnz)
print("Sample item_id:", item_ids[0])
print("First 10 terms:", feature_names[:10])

TF-IDF matrix shape: (927, 18549)
Number of non-zero entries: 121649
Sample item_id: B00000JRSB
First 10 terms: ['00' '000' '007' '03' '06' '065' '07' '073' '074' '08']


In [14]:
# represent each item using pretrained word embeddings, we will use spacy for this task
# create a new column in the processed_metadata_df for the item representation
def get_item_representation(row):
    title = row['title'] if isinstance(row['title'], str) else ''
    description = row['processed_description'] if isinstance(row['processed_description'], str) else ''
    text = f"{title} {description}".strip()
    doc = nlp(text)
    return doc.vector

processed_metadata_df['item_representation'] = processed_metadata_df.apply(get_item_representation, axis=1)
processed_metadata_df.head(3)

,item_id,title,description,features,categories,details,rating_number,average_rating,processed_description,combined_text,item_representation
864,B00000JRSB,Final Fantasy VII - PlayStation,"[Amazon.com, Long recognized as role-playing g...","[1 Player, RPG, 3 Disc Set, Excellent graphics...","[Video Games, Legacy Systems, PlayStation Syst...","{'': None, 'AC Adapter Current': None, 'Access...",1631,4.7,amazoncom long recognized roleplaying games pa...,Final Fantasy VII - PlayStation amazoncom long...,"[-0.69108784, 0.10298124, -0.098285764, -0.047..."
1312,B00001X50M,Metal Gear Solid,"[Product description, You are Snake, a governm...","[Lightly Armed and facing an army of foes, Sna...","[Video Games, Legacy Systems, PlayStation Syst...","{'': None, 'AC Adapter Current': None, 'Access...",638,4.6,product description snake government agent mis...,Metal Gear Solid product description snake gov...,"[-0.6971409, 0.058839068, -0.09635035, -0.0459..."
1354,B00001XDUB,Silent Hill,"[Product Description, In, Silent Hill, , you a...",[],"[Video Games, Legacy Systems, PlayStation Syst...","{'': None, 'AC Adapter Current': None, 'Access...",399,4.4,product description silent hill assume role wi...,Silent Hill product description silent hill as...,"[-0.6877667, 0.10404825, -0.11832558, -0.08161..."


In [17]:
# Explore the similarity between items within the vector spaces by computing their cosine similarity. 
# compare between a subset of the items do like 50 items
from sklearn.metrics.pairwise import cosine_similarity
# comparing tf-idf representations
# smaller subset 

preview_subset_size = 50
subset_titles = (
    processed_metadata_df["title"]
    .fillna("<missing title>")
    .astype(str)
    .iloc[:preview_subset_size]
    .tolist()
)

tfidf_sim_preview = cosine_similarity(tfidf_matrix[:preview_subset_size])
embedding_sim_preview = cosine_similarity(
    processed_metadata_df["item_representation"].iloc[:preview_subset_size].tolist()
)

def show_low_high_pairs(sim_matrix, titles, label, top_k=5):
    pairs = []
    n = sim_matrix.shape[0]
    for i in range(n):
        for j in range(i + 1, n):
            pairs.append((float(sim_matrix[i, j]), i, j))

    pairs_sorted = sorted(pairs, key=lambda x: x[0])

    print(f"\n{label} — Lowest {top_k} similarity pairs:")
    for score, i, j in pairs_sorted[:top_k]:
        print(f"{score:.4f} | {titles[i]}  <->  {titles[j]}")

    print(f"\n{label} — Highest {top_k} similarity pairs:")
    for score, i, j in pairs_sorted[-top_k:][::-1]:
        print(f"{score:.4f} | {titles[i]}  <->  {titles[j]}")

show_low_high_pairs(tfidf_sim_preview, subset_titles, "TF-IDF", top_k=5)
show_low_high_pairs(embedding_sim_preview, subset_titles, "Embeddings", top_k=5)
subset_size = 10
subset_tfidf = tfidf_matrix[:subset_size]
similarity_matrix = cosine_similarity(subset_tfidf)
# compare word embedding representations
subset_embeddings = processed_metadata_df['item_representation'].tolist()[:subset_size]
similarity_matrix_embeddings = cosine_similarity(subset_embeddings) # what does this do?
# it computes the cosine similarity between all pairs of item representations in the subset, 
# resulting in a similarity matrix where entry (i, j) represents the cosine similarity between item i and item j
# based on their word embedding representations.
# display results


TF-IDF — Lowest 5 similarity pairs:
0.0000 | Metal Gear Solid  <->  Final Fantasy IX
0.0000 | Metal Gear Solid  <->  Sony PlayStation 3 Blu-ray Disc Remote
0.0000 | Silent Hill  <->  Final Fantasy IX
0.0000 | Silent Hill  <->  Metal Gear Solid 2: Sons of Liberty - PlayStation 2
0.0000 | Silent Hill  <->  Sony PlayStation 3 Blu-ray Disc Remote

TF-IDF — Highest 5 similarity pairs:
0.4427 | Final Fantasy VII - PlayStation  <->  Final Fantasy VIII
0.4139 | Final Fantasy VIII  <->  Final Fantasy IX
0.4134 | Super Mario Sunshine  <->  New Super Mario Bros
0.4126 | Final Fantasy IX  <->  Final Fantasy Origins Final Fantasy I & II Remastered Editions - PlayStation
0.4118 | God of War - PlayStation 2  <->  God of War 2 - PlayStation 2

Embeddings — Lowest 5 similarity pairs:
0.2709 | Final Fantasy IX  <->  Sony PlayStation 3 Blu-ray Disc Remote
0.3977 | Final Fantasy IX  <->  Metal Gear Solid 2: Sons of Liberty - PlayStation 2
0.4009 | Final Fantasy X-2  <->  Sony PlayStation 3 Blu-ray Disc R